In [2]:
import numpy as np
import scipy
import os
import time
from tqdm import tqdm
import pandas as pd
sig_length = 20

In [2]:
allParams = []
with open("Variable_Documentation.txt", "r", encoding="utf-8") as file:
    lines = file.readlines()

for line in lines:
    allParams.append(line.split(' ')[0])

In [3]:
givenParams = ['WOW', 'LATP', 'LONP', 'BAL1', 'ROLL', 'PTCH', 'IVV', 'GS', 'CAS', 'VRTG', 'LATG', 'FLAP', 'PLA_2', 'PLA_3', 'PLA_4', 'TRK', 'TH', 'WS', 'WD', 'TAT', 'SAT', 'LOC', 'GLS']

In [4]:
sigLength = 20

In [5]:
def find_indexWOW(arr):
    left, right = 0, len(arr) - 1
    idx = -1
    while left <= right:
        mid = left + (right - left) // 2
        if arr[mid] == 1:
            left = mid + 1
        elif arr[mid] == 0:
            if mid > 0 and arr[mid - 1] == 1:
                idx = mid
                break
            if arr[:mid].sum() == 0:
                left = mid + 1
            else:
                right = mid - 1
    
    if arr[idx:].sum() == 0:
        return idx
    return find_indexWOW(arr[idx+1:]) + idx + 1

In [11]:
import os
import scipy.io
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Define necessary paths and parameters
dataset_folder = "C:/Users/HP/Desktop/ML/project"
folder_list = [folder for folder in os.listdir(dataset_folder) if "Tail" in folder]

required_data = []
timestamps = []

# Define a function to process each timestamp
def process_timestamp(flight_path, timestamp, givenParams, sigLength):
    try:
        # Load the .mat file
        data = scipy.io.loadmat(os.path.join(flight_path, timestamp))
    except:
        return None, None  # Skip files that can't be loaded

    # Find length and touchdown index
    length = len(data['WOW'][0][0][0])
    try:
        touch_down_index = find_indexWOW(data['WOW'][0][0][0])
    except:
        # Fallback for finding touch down index if find_index fails
        temp_data = data['WOW'][0][0][0][::-1]
        idx = next((i for i, val in enumerate(temp_data) if val == 1), -1)
        touch_down_index = length - 1 - idx if idx != -1 else None
        if touch_down_index is None:
            return None, None

        # Extract and downsample data
    features = []
    for param in givenParams:
        signal = data[param][0][0][0].flatten()
        length_signal = len(signal)
        signal_mul_factor = length_signal // length
        down_sampled_signal = np.mean(signal.reshape(-1, signal_mul_factor), axis=1)
        required_signal = down_sampled_signal[touch_down_index - sigLength : touch_down_index + 1]
        
        if len(required_signal) != sigLength + 1:
            return None, None  # skip if the segment is incomplete

        # Compute mean and std of the 20 timesteps
        mean_val = np.mean(required_signal)
        std_val = np.std(required_signal)

        features.extend([mean_val, std_val])  # add both to final feature list
    flight_data = np.array(features)
    return flight_data.flatten() if flight_data.size > 0 else None, timestamp

# Main processing function
def process_flight(flight_idx, flight, givenParams, sigLength):
    flight_path = os.path.join(dataset_folder, flight)
    flight_data_list = []

    with ThreadPoolExecutor(max_workers=20) as executor:
        futures = {
            executor.submit(process_timestamp, flight_path, timestamp, givenParams, sigLength): timestamp
            for timestamp in os.listdir(flight_path)
        }
        
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"Flight {flight_idx}"):
            flight_data, timestamp = future.result()
            if flight_data is not None:
                flight_data_list.append((flight_data, timestamp))

    return flight_data_list

# Aggregate data for all flights
all_flight_data = []
for flight_idx, flight in enumerate(folder_list):
    flight_data_list = process_flight(flight_idx, flight, givenParams, sigLength)
    for data, timestamp in flight_data_list:
        required_data.append(data)
        timestamps.append(timestamp)

Flight 5: 100%|██████████| 606/606 [00:32<00:00, 18.49it/s]


In [12]:
len(required_data)

3219

In [13]:
r = []
t = []
for i in range(len(required_data)):
    if len(required_data[i]) == 46:
        r.append(required_data[i])
        t.append(timestamps[i])
required_data = r
timestamps = t

In [14]:
required_data = np.array(required_data)

In [15]:
required_data.shape

(3219, 46)

In [20]:
var_names = [f"{param}_mean" for param in givenParams] + [f"{param}_std" for param in givenParams] + ["TouchdownDistance"]

In [21]:
len(var_names)

47

In [22]:
touchdown_data = pd.read_csv("LongLandingAnalysis.csv", index_col=0)
touchdown_data.set_index("Timestamps", inplace=True)

In [23]:
available_indices = []
for i in range(len(timestamps)):
    if timestamps[i] in touchdown_data.index:
        available_indices.append(i)

In [24]:
len(available_indices)

2095

In [25]:
t = []
for i in available_indices:
    t.append(timestamps[i])
r = required_data[available_indices]

timestamps = t
required_data = r

In [26]:
touchdown_distances = touchdown_data.loc[timestamps, "TouchdownDistance"].to_numpy()
required_data = np.c_[required_data, touchdown_distances]

In [27]:
len(required_data)

2095

In [28]:
# save data
np.save("required_data.npy", required_data)

timestamps_save_file = "timestamps_save_file.txt"
with open(timestamps_save_file, "w") as f:
    for item in timestamps:
        f.write(f"{item}\n")

In [25]:
np.save("required_data_future.npy", required_data)

In [17]:
newArrTimestepsMorePrior = np.load('required_data2 (1).npy')

In [18]:
newArrTimestepsMorePrior.shape
dfTimestepsMorePrior = pd.DataFrame(newArrTimestepsMorePrior)
dfTimestepsMorePrior.rename(columns={0: 'Timestamps', 462: 'isLongLanding', 461: 'TouchdownDistance'}, inplace=True)
dfTimestepsMorePrior.head()

,Timestamps,1,2,3,4,5,6,7,8,9,...,453,454,455,456,457,458,459,460,TouchdownDistance,isLongLanding
0,664200204041304.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.02379000000655651,0.032760001718997955,0.042899999767541885,0.04836000129580498,0.05226000025868416,0.06552000343799591,0.06552000343799591,0.06435000151395798,3049.2442707931527,0.0
1,664200206140348.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.026130022792816177,-0.03003001430511476,-0.031199993877410903,-0.026520015983581557,-0.012870015888214126,0.0007800000021234155,0.015990000218153,0.02027999982237816,1334.0703522572776,0.0
2,664200206091307.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.014039995460510268,-0.02730000236511232,-0.03744000413894655,-0.03860998371124269,-0.02691000917434694,-0.013260009078979507,-0.0058500192451477195,-0.0058500192451477195,2173.349815306356,0.0
3,664200206041253.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.03042000749588014,-0.031589987068176284,-0.032369973449707046,-0.03003001430511476,-0.034319999008178725,-0.03860998371124269,-0.030810000686645522,-0.02964002111434938,1064.1374718099312,0.0
4,664200206121648.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.010529997138977065,-0.008970024375915542,-0.0007799885559082176,-0.0019500277328491356,0.0,0.012090000323951244,0.02769000083208084,0.02184000052511692,1553.4620396224484,0.0


In [19]:
def func(string):
    return string.split('.')[0]
LongLandingAnalaysis = pd.read_csv('LongLandingAnalysis.csv')
runway_df = pd.read_csv("runway_df.csv")
runway_df.rename(columns={'Airport': 'NearestAirport', 'Ident': 'NearestRunway'}, inplace=True)
# LongLandingAnalaysis['Timestamps'].dtype
#dfTimestepsMorePrior['Timestamps'].dtype
# MagneticHeadingDataLanding['Timestamps'] = MagneticHeadingDataLanding['Timestamps'].astype(np.int64)
dfTimestepsMorePriorMerged =dfTimestepsMorePrior.merge(LongLandingAnalaysis, on='Timestamps', how='left')
dfTimestepsMorePriorMerged =dfTimestepsMorePriorMerged.merge(runway_df, on=['NearestAirport', 'NearestRunway'], how='left')
dfTimestepsMorePriorMerged.head()

,Timestamps,1,2,3,4,5,6,7,8,9,...,FORMATTED LONGITUDE,GOOGLE EARTH,ThrDisp,Elevation,ILS Navaid,ILS Cat,ILS LBearing,ThrCrossHeight,GPIP_Offset(Assuming 3 degree glideslope),combined_coordinates
0,664200204041304.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,083 21 00.00W,42 13 52.41N 083 21 00.00W,,636,IDWC,2.0,216.0,60,NaN,"[42.231225, -83.35]"
1,664200206140348.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,093 11 40.69W,44 52 53.53N 093 11 40.69W,200,820,IINN,1.0,301.0,55,NaN,"[44.88153611111111, -93.19463611111111]"
2,664200206091307.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,097 35 20.16W,35 22 41.63N 097 35 20.16W,,1283,IRGR,2.0,356.0,56,NaN,"[35.378230555555554, -97.58893333333333]"
3,664200206041253.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,089 03 43.50W,30 23 50.91N 089 03 43.50W,,22,IUXI,1.0,317.0,55,NaN,"[30.397475, -89.06208333333333]"
4,664200206121648.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,091 15 31.97W,43 53 35.72N 091 15 31.97W,,653,ILSE,1.0,180.0,55,NaN,"[43.893255555555555, -91.25888055555555]"


In [20]:
dfTimestepsMorePriorMerged = dfTimestepsMorePriorMerged.loc[~((dfTimestepsMorePriorMerged['NearestRunway'] == 'RW10L') & (dfTimestepsMorePriorMerged['NearestAirport'] == 'KORD'))]
dfTimestepsMorePriorMerged = dfTimestepsMorePriorMerged.loc[~((dfTimestepsMorePriorMerged['NearestRunway'] == 'RW24') & (dfTimestepsMorePriorMerged['NearestAirport'] == 'KBHM'))]
dfTimestepsMorePriorMerged = dfTimestepsMorePriorMerged.loc[~((dfTimestepsMorePriorMerged['NearestRunway'] == 'RW13') & (dfTimestepsMorePriorMerged['NearestAirport'] == 'KRST'))]
dfTimestepsMorePriorMerged = dfTimestepsMorePriorMerged.loc[~((dfTimestepsMorePriorMerged['NearestRunway'] == 'RW04') & (dfTimestepsMorePriorMerged['NearestAirport'] == 'KLEX'))]
dfTimestepsMorePriorMerged = dfTimestepsMorePriorMerged.loc[~((dfTimestepsMorePriorMerged['NearestRunway'] == 'RW35') & (dfTimestepsMorePriorMerged['NearestAirport'] == 'KAZO'))]
dfTimestepsMorePriorMerged = dfTimestepsMorePriorMerged.loc[~((dfTimestepsMorePriorMerged['NearestRunway'] == 'RW16') & (dfTimestepsMorePriorMerged['NearestAirport'] == 'KHPN'))]
dfTimestepsMorePriorMerged = dfTimestepsMorePriorMerged.loc[~((dfTimestepsMorePriorMerged['NearestRunway'] == 'RW17') & (dfTimestepsMorePriorMerged['NearestAirport'] == 'KPNS'))]
dfTimestepsMorePriorMerged = dfTimestepsMorePriorMerged.loc[~((dfTimestepsMorePriorMerged['NearestRunway'] == 'RW28') & (dfTimestepsMorePriorMerged['NearestAirport'] == 'KTVC'))]
dfTimestepsMorePriorMerged = dfTimestepsMorePriorMerged.loc[~((dfTimestepsMorePriorMerged['NearestRunway'] == 'RW35L') & (dfTimestepsMorePriorMerged['NearestAirport'] == 'KSDF'))]
dfTimestepsMorePriorMerged = dfTimestepsMorePriorMerged.loc[~((dfTimestepsMorePriorMerged['NearestRunway'] == 'RW17C') & (dfTimestepsMorePriorMerged['NearestAirport'] == 'KDFW'))]

In [21]:
cols = dfTimestepsMorePriorMerged.columns.tolist()[:463] + ['RunwayLength', 'Latitude_y', 'Longitude_y', 'Elevation', 'ThrCrossHeight', 'MagneticHeadingMeanUpdated', 'MagBearing']
dfTimestepsMorePriorMerged = dfTimestepsMorePriorMerged[cols]
dfTimestepsMorePriorMerged.rename(columns={'TouchdownDistance_x': 'TouchdownDistance', 'Latitude_y': 'Latitude', 'Longitude_y': 'Longitude'}, inplace=True)

In [22]:
dfTimestepsMorePriorMerged['MagneticAlignment'] = abs(dfTimestepsMorePriorMerged['MagneticHeadingMeanUpdated'] - dfTimestepsMorePriorMerged['MagBearing'])
dfTimestepsMorePriorMerged.drop(columns = ['MagneticHeadingMeanUpdated', 'MagBearing'], inplace=True)
dfTimestepsMorePriorMerged.head()

,Timestamps,1,2,3,4,5,6,7,8,9,...,459,460,TouchdownDistance,isLongLanding_x,RunwayLength,Latitude,Longitude,Elevation,ThrCrossHeight,MagneticAlignment
0,664200204041304.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.06552000343799591,0.06435000151395798,3049.2442707931527,0.0,12003,42.231225,-83.350000,636,60,1.265820
1,664200206140348.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.015990000218153,0.02027999982237816,1334.0703522572776,0.0,8200,44.881536,-93.194636,820,55,2.348572
2,664200206091307.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.0058500192451477195,-0.0058500192451477195,2173.349815306356,0.0,9803,35.378231,-97.588933,1283,56,2.266140
3,664200206041253.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.030810000686645522,-0.02964002111434938,1064.1374718099312,0.0,9002,30.397475,-89.062083,22,55,3.671962
4,664200206121648.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.02769000083208084,0.02184000052511692,1553.4620396224484,0.0,8742,43.893256,-91.258881,653,55,1.527618


In [23]:
touchdown_distances = dfTimestepsMorePriorMerged['TouchdownDistance']
isLongLandingRecords = dfTimestepsMorePriorMerged['isLongLanding_x']
dfTimestepsMorePriorMerged.drop(columns = ['TouchdownDistance', 'isLongLanding_x'], inplace=True)
dfTimestepsMorePriorMerged.head()

,Timestamps,1,2,3,4,5,6,7,8,9,...,457,458,459,460,RunwayLength,Latitude,Longitude,Elevation,ThrCrossHeight,MagneticAlignment
0,664200204041304.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.05226000025868416,0.06552000343799591,0.06552000343799591,0.06435000151395798,12003,42.231225,-83.350000,636,60,1.265820
1,664200206140348.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.012870015888214126,0.0007800000021234155,0.015990000218153,0.02027999982237816,8200,44.881536,-93.194636,820,55,2.348572
2,664200206091307.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.02691000917434694,-0.013260009078979507,-0.0058500192451477195,-0.0058500192451477195,9803,35.378231,-97.588933,1283,56,2.266140
3,664200206041253.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.034319999008178725,-0.03860998371124269,-0.030810000686645522,-0.02964002111434938,9002,30.397475,-89.062083,22,55,3.671962
4,664200206121648.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.012090000323951244,0.02769000083208084,0.02184000052511692,8742,43.893256,-91.258881,653,55,1.527618


In [24]:
dfTimestepsMorePriorMerged['TouchdownDistance'] = touchdown_distances
dfTimestepsMorePriorMergedNumpy = dfTimestepsMorePriorMerged.to_numpy()
np.save("dfTimestepsMorePriorMergedNumpy.npy", dfTimestepsMorePriorMergedNumpy)

In [25]:
dfTimestepsMorePriorMerged.drop(columns = {'TouchdownDistance'}, inplace=True)
dfTimestepsMorePriorMerged['isLongLanding'] = isLongLandingRecords
dfTimestepsMorePriorMergedLLNumpy = dfTimestepsMorePriorMerged.to_numpy()
np.save("dfTimestepsMorePriorMergedLLNumpy.npy", dfTimestepsMorePriorMergedLLNumpy)

In [27]:
dfTimestepsMorePriorMerged.shape

(24795, 468)

__PPD New Approach__

In [23]:
def helperFunction1(pathToNPY, givenParams):
    arr = np.load(pathToNPY)
    print(arr.shape)
    # columns 3 to 485 are numeric, can convert to float
    timeSeriesData = arr[:, 2:485].astype(float)
    # mean = timeSeriesData.mean(axis = 2)
    # std = timeSeriesData.std(axis = 2)
    # finalFeatures = np.concatenate([mean, std], axis=1)
    # var_names = [f"{param}_mean" for param in givenParams] + [f"{param}_std" for param in givenParams]
    
    # Create a dataframe from this numpy array
    df = pd.DataFrame(data = timeSeriesData)
    df['Timestamps'] = arr[:,0]
    cols = df.columns.tolist()
    cols.insert(0, cols.pop(cols.index('Timestamps')))
    df = df[cols]
    df['TouchdownDistance'] = arr[:, 485].astype(float)
    
    return df

In [24]:
givenParams = ['WOW', 'LATP', 'LONP', 'BAL1', 'ROLL', 'PTCH', 'IVV', 'GS', 'CAS', 'VRTG', 'LATG', 'FLAP', 'PLA_2', 'PLA_3', 'PLA_4', 'TRK', 'TH', 'WS', 'WD', 'TAT', 'SAT', 'LOC', 'GLS']
requiredDataWithTimestamps = helperFunction1('requiredDataWithTimestamps.npy', givenParams)

(24339, 489)


In [25]:
def func(string):
    return string.split('.')[0]
LongLandingAnalaysis = pd.read_csv('LongLandingAnalysis.csv')
runway_df = pd.read_csv("runway_df.csv")
runway_df.rename(columns={'Airport': 'NearestAirport', 'Ident': 'NearestRunway'}, inplace=True)
LongLandingAnalaysis['Timestamps'].dtype
requiredDataWithTimestamps['Timestamps'].dtype
# MagneticHeadingDataLanding['Timestamps'] = MagneticHeadingDataLanding['Timestamps'].astype(np.int64)
requiredDataWithTimestampsMerged = requiredDataWithTimestamps.merge(LongLandingAnalaysis, on='Timestamps', how='left')
requiredDataWithTimestampsMerged = requiredDataWithTimestampsMerged.merge(runway_df, on=['NearestAirport', 'NearestRunway'], how='left')
requiredDataWithTimestampsMerged.head()


,Timestamps,0,1,2,3,4,5,6,7,8,...,FORMATTED LONGITUDE,GOOGLE EARTH,ThrDisp,Elevation,ILS Navaid,ILS Cat,ILS LBearing,ThrCrossHeight,GPIP_Offset(Assuming 3 degree glideslope),combined_coordinates
0,659200111241351.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,096 48 59.06W,46 54 29.53N 096 48 59.06W,,899.0,IFAR,1.0,356.0,55.0,NaN,"[46.908202777777774, -96.81640555555555]"
1,659200112271134.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,089 57 21.09W,35 03 28.03N 089 57 21.09W,,292.0,IJIM,1.0,273.0,54.0,NaN,"[35.057786111111106, -89.95585833333334]"
2,659200202131023.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,083 21 58.59W,42 13 34.50N 083 21 58.59W,,642.0,IJKI,2.0,216.0,55.0,NaN,"[42.22625, -83.36627499999999]"
3,659200202121243.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,084 35 56.34W,38 02 31.50N 084 35 56.34W,,973.0,IGNJ,1.0,226.0,56.0,NaN,"[38.04208333333333, -84.59898333333332]"
4,659200112111314.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,097 01 33.47W,32 54 56.53N 097 01 33.47W,,562.0,IFLQ,3.0,176.0,58.0,NaN,"[32.915702777777774, -97.02596388888888]"


In [26]:
cols = requiredDataWithTimestampsMerged.columns.tolist()[:485] + ['Length', 'Latitude_y', 'Longitude_y', 'Elevation', 'ThrCrossHeight', 'MagneticHeadingMeanUpdated', 'MagBearing']
requiredDataWithTimestampsMerged = requiredDataWithTimestampsMerged[cols]
requiredDataWithTimestampsMerged.rename(columns={'TouchdownDistance_x': 'TouchdownDistance', 'Latitude_y': 'Latitude', 'Longitude_y': 'Longitude'}, inplace=True)

In [27]:
requiredDataWithTimestampsMerged['TouchdownDist'] = requiredDataWithTimestampsMerged['TouchdownDistance']
requiredDataWithTimestampsMerged.drop(columns=['TouchdownDistance'], inplace= True)

In [28]:
requiredDataWithTimestampsMerged.rename(columns={'TouchdownDist' : 'TouchdownDistance'}, inplace=True)

In [30]:
requiredDataWithTimestampsMerged['MagneticAlignment'] = abs(requiredDataWithTimestampsMerged['MagneticHeadingMeanUpdated'] - requiredDataWithTimestampsMerged['MagBearing'])
requiredDataWithTimestampsMerged.drop(columns = ['MagneticHeadingMeanUpdated', 'MagBearing'], inplace=True)
requiredDataWithTimestampsMerged.head()

,Timestamps,0,1,2,3,4,5,6,7,8,...,480,481,482,Length,Latitude,Longitude,Elevation,ThrCrossHeight,TouchdownDistance,MagneticAlignment
0,659200111241351.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.18876,-0.36036,-0.49296,9001.0,46.908203,-96.816406,899.0,55.0,1639.915003,1.798549
1,659200112271134.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.00195,0.06357,0.06279,8946.0,35.057786,-89.955858,292.0,54.0,1405.330113,3.253425
2,659200202131023.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.58539,-0.20241,-0.17043,10000.0,42.226250,-83.366275,642.0,55.0,1320.815633,2.104880
3,659200202121243.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.57213,-0.42900,-0.51792,7004.0,38.042083,-84.598983,973.0,56.0,1514.608794,1.757441
4,659200112111314.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.30147,0.20046,0.12870,13401.0,32.915703,-97.025964,562.0,58.0,5128.627886,1.743496


In [33]:
cols = requiredDataWithTimestampsMerged.columns.tolist()[:-2] + ['MagneticAlignment', 'TouchdownDistance']
requiredDataWithTimestampsMerged = requiredDataWithTimestampsMerged[cols]

In [37]:
requiredDataWithTimestampsMergedNumpy = requiredDataWithTimestampsMerged.to_numpy()
np.save("requiredDataWithTimestampsMergedNumpy.npy", requiredDataWithTimestampsMergedNumpy)

In [13]:
def helperFunction2(pathToNPY, givenParams):
    var_names = []
    
    arr = np.load(pathToNPY)
    print(arr.shape)
    timeSeriesData = arr[:, 2:485].astype(float)
    # timeSeriesData = arr[:, 1:-1].astype(float)
    # for param in givenParams:
    #     var_names.extend([f"{param}_mean", f"{param}_std"])
    
    # Create a dataframe from this numpy array
    df = pd.DataFrame(data = timeSeriesData)
    df['Timestamps'] = arr[:,0]
    cols = df.columns.tolist()
    cols.insert(0, cols.pop(cols.index('Timestamps')))
    df = df[cols]
    df['TouchdownDistance'] = arr[:, -1].astype(float)
    
    return df

In [14]:
givenParams = ['WOW', 'LATP', 'LONP', 'BAL1', 'ROLL', 'PTCH', 'IVV', 'GS', 'CAS', 'VRTG', 'LATG', 'FLAP', 'PLA_2', 'PLA_3', 'PLA_4', 'TRK', 'TH', 'WS', 'WD', 'TAT', 'SAT', 'LOC', 'GLS']
requiredDataWithTimestamps2 = helperFunction2('requiredDataWithTimestamps2.npy', givenParams)

(23163, 48)


In [6]:
requiredDataWithTimestamps2

,Timestamps,WOW_mean,WOW_std,LATP_mean,LATP_std,LONP_mean,LONP_std,BAL1_mean,BAL1_std,ROLL_mean,...,WD_std,TAT_mean,TAT_std,SAT_mean,SAT_std,LOC_mean,LOC_std,GLS_mean,GLS_std,TouchdownDistance
0,652200101231656.mat,1.0,0.0,42.969305,0.000081,-83.732361,0.004541,803.285714,43.065446,0.170479,...,14.268629,29.630952,0.165814,27.773810,0.369240,-0.002856,0.003210,0.135646,0.119513,1943.047891
1,652200107111347.mat,1.0,0.0,43.915118,0.002748,-92.493652,0.002002,1356.702381,58.191782,-0.367181,...,17.063271,28.392857,0.182108,26.369048,0.382808,-0.015857,0.024661,-0.012684,0.029171,1156.009495
2,652200101240601.mat,1.0,0.0,42.200630,0.002742,-83.372410,0.001886,671.928571,50.139516,-0.395366,...,7.354573,22.190476,0.216915,20.428571,0.409288,0.000644,0.002413,-0.031961,0.059451,773.446274
3,652200106291349.mat,1.0,0.0,46.841823,0.000113,-92.177929,0.004776,1458.083333,60.081608,0.568918,...,15.717423,13.166667,0.281718,11.059524,0.522726,-0.010733,0.025326,-0.012201,0.029454,1556.577668
4,652200107180523.mat,1.0,0.0,35.058200,0.000147,-89.955126,0.004138,320.083333,57.707974,-0.162698,...,79.717386,18.297619,0.182884,16.476190,0.131492,0.002081,0.007130,0.065557,0.041546,1967.280727
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23158,654200105240505.mat,1.0,0.0,44.880517,0.001736,-93.193424,0.004053,854.071429,55.651399,0.036358,...,127.620408,22.369048,0.847331,20.559524,0.981692,0.002847,0.007279,0.012239,0.127463,1686.962875
23159,654200106030906.mat,1.0,0.0,38.738497,0.001750,-90.340726,0.003572,627.250000,51.348019,-0.836179,...,61.364591,26.357143,0.543202,24.535714,0.760974,-0.004620,0.003052,-0.135534,0.263020,2354.334861
23160,654200103231610.mat,1.0,0.0,44.894250,0.001675,-93.219007,0.003813,847.630952,42.946785,0.468409,...,12.498121,32.809524,0.317212,30.857143,0.323407,-0.000737,0.017383,0.038257,0.125452,2322.925294
23161,654200106120652.mat,1.0,0.0,43.643068,0.000339,-70.299495,0.004485,95.869048,47.602935,0.117249,...,46.594100,22.476190,0.576859,20.619048,0.750472,-0.000355,0.003583,-0.015191,0.163382,2166.336349


In [7]:
var_names = ['Timestamps'] + [f"{param}_mean" for param in givenParams] + [f"{param}_std" for param in givenParams] + ['TouchdownDistance']
requiredDataWithTimestamps2 = requiredDataWithTimestamps2[var_names]

In [8]:
finalMergedRequiredData = pd.concat([requiredDataWithTimestamps, requiredDataWithTimestamps2], axis = 0, ignore_index=True)

In [38]:
LongLandingAnalysisGeoPy = pd.read_csv('LongLandingAnalysis.csv')

In [39]:
LongLandingAnalysisGeoPy.shape

(74248, 15)

In [40]:
mergedDf = pd.merge(requiredDataWithTimestampsMerged, LongLandingAnalysisGeoPy, on='Timestamps', how='inner')

In [41]:
mergedDf.shape

(24338, 505)

In [42]:
mergedDf = mergedDf.loc[~((mergedDf['NearestRunway'] == 'RW10L') & (mergedDf['NearestAirport'] == 'KORD'))]
mergedDf = mergedDf.loc[~((mergedDf['NearestRunway'] == 'RW24') & (mergedDf['NearestAirport'] == 'KBHM'))]
mergedDf = mergedDf.loc[~((mergedDf['NearestRunway'] == 'RW13') & (mergedDf['NearestAirport'] == 'KRST'))]
mergedDf = mergedDf.loc[~((mergedDf['NearestRunway'] == 'RW04') & (mergedDf['NearestAirport'] == 'KLEX'))]
mergedDf = mergedDf.loc[~((mergedDf['NearestRunway'] == 'RW35') & (mergedDf['NearestAirport'] == 'KAZO'))]
mergedDf = mergedDf.loc[~((mergedDf['NearestRunway'] == 'RW16') & (mergedDf['NearestAirport'] == 'KHPN'))]
mergedDf = mergedDf.loc[~((mergedDf['NearestRunway'] == 'RW17') & (mergedDf['NearestAirport'] == 'KPNS'))]
mergedDf = mergedDf.loc[~((mergedDf['NearestRunway'] == 'RW28') & (mergedDf['NearestAirport'] == 'KTVC'))]
mergedDf = mergedDf.loc[~((mergedDf['NearestRunway'] == 'RW35L') & (mergedDf['NearestAirport'] == 'KSDF'))]
mergedDf = mergedDf.loc[~((mergedDf['NearestRunway'] == 'RW17C') & (mergedDf['NearestAirport'] == 'KDFW'))]

In [43]:
mergedDf.columns

Index([                'Timestamps',                            0,
                                  1,                            2,
                                  3,                            4,
                                  5,                            6,
                                  7,                            8,
       ...
                  'MagneticHeading',        'MagneticHeadingMean',
       'MagneticHeadingMeanUpdated',                     'MinIdx',
                    'NearestRunway',             'NearestAirport',
                    'AirportCoords',               'RunwayLength',
                    'isLongLanding',        'TouchdownDistance_y'],
      dtype='object', length=505)

In [44]:
requiredCols = mergedDf.columns.to_list()[0:491]
requiredCols.append('isLongLanding')
mergedDf = mergedDf[requiredCols]
mergedDf.head()

,Timestamps,0,1,2,3,4,5,6,7,8,...,481,482,Length,Latitude_x,Longitude_x,Elevation,ThrCrossHeight,MagneticAlignment,TouchdownDistance_x,isLongLanding
0,659200111241351.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.36036,-0.49296,9001.0,46.908203,-96.816406,899.0,55.0,1.798549,1639.915003,0.0
1,659200112271134.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.06357,0.06279,8946.0,35.057786,-89.955858,292.0,54.0,3.253425,1405.330113,0.0
2,659200202131023.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.20241,-0.17043,10000.0,42.226250,-83.366275,642.0,55.0,2.104880,1320.815633,0.0
3,659200202121243.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.42900,-0.51792,7004.0,38.042083,-84.598983,973.0,56.0,1.757441,1514.608794,0.0
5,659200201281729.mat,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,-0.34710,-0.15288,9003.0,41.537942,-93.650572,935.0,56.0,3.768421,2067.536266,0.0


In [45]:
mergedDf.drop(columns=['TouchdownDistance_x'], inplace=True)

In [26]:
mergedDf.rename(columns={'TouchdownDistance_x':'TouchdownDistance'}, inplace=True)

In [46]:
mergedDf.shape

(23452, 491)

In [47]:
numpyArray = mergedDf.to_numpy()

# Define the filename
filename = 'requiredDataWithLongLandingsMerged.npy'

# Save the NumPy array to the file
np.save(filename, numpyArray)

In [39]:
requiredDataWithLongLandings = np.load('requiredDataWithLongLandings.npy', allow_pickle=True)
requiredDataWithLongLandingsArray = np.hstack([requiredDataWithLongLandings[:, 1:484], requiredDataWithLongLandings[:, [-1]]])

In [41]:
# Define the filename
filename = 'requiredDataWithLongLandingsArray.npy'

# Save the NumPy array to the file
np.save(filename, requiredDataWithLongLandingsArray)

In [40]:
requiredDataWithLongLandingsArray.shape

(23452, 484)

In [27]:
mergedDf.to_csv('FullFilteredRequiredData.csv', index= False)

In [29]:
mergedDf['isLongLanding'].value_counts()

isLongLanding
0.0    44014
1.0     1772
Name: count, dtype: int64

In [31]:
(1772/44014)*100

4.0259917299041215